# First-Stage Environment Exfiltration Dataflow

> **Safety boundary:** This workbook performs static analysis of sanitized excerpts, fake fixtures, and derived public metadata. It does not execute attacker code, inspect the analyst's environment, or contact any network.

## Data Flow

```text
project configuration
  -> dotenv-style loader
  -> process environment
  -> bulk object copy
  -> POST request body
  -> serverless endpoint
  -> JavaScript response
```

The dangerous primitive is not one particular API key. It is bulk capture of a
developer process environment, where cloud credentials, package tokens,
service URLs, and shell context commonly coexist.

The published first-stage helper spread `process.env` into the POST body and
used the request marker `x-app-request: ip-check`. The exact source remains in
the incident report; executable cells below use only fake values.

In [ ]:
import json
from pathlib import Path

fake_env = json.loads(Path("../fixtures/fake-process-env.redacted.json").read_text())

def category(key):
    if key.startswith("AWS_"):
        return "cloud"
    if "ALCHEMY" in key:
        return "blockchain/API"
    if "AUTH" in key:
        return "auth/session"
    if key.startswith("npm_"):
        return "npm lifecycle"
    if key in {"HOME", "PATH"}:
        return "path/home"
    return "local runtime"

rows = [
    {
        "env_key": key,
        "category": category(key),
        "why_it_matters": {
            "cloud": "may authorize cloud API operations",
            "blockchain/API": "may identify or authorize a service account",
            "auth/session": "may expose service authentication configuration",
            "npm lifecycle": "reveals execution context and triggering hook",
            "path/home": "reveals host layout and command search path",
            "local runtime": "describes application mode",
        }[category(key)],
        "real_value_published": False,
    }
    for key in fake_env
]
rows

| Env key | Category | Why it matters | Real value published? |
|---|---|---|---|
| `AWS_ACCESS_KEY_ID` | cloud | may identify cloud principal | no |
| `AWS_SECRET_ACCESS_KEY` | cloud | may authorize cloud API operations | no |
| `ALCHEMY_API_KEY` | blockchain/API | may authorize or identify a service account | no |
| `AUTH_API` | auth/session | identifies authentication configuration | no |
| `npm_lifecycle_event` | npm lifecycle | confirms execution context | no real capture |
| `HOME`, `PATH` | path/home | describes host layout | fake examples only |

In [ ]:
fake_request = {
    "method": "POST",
    "path": "/api",
    "headers": {
        "content-type": "application/json",
        "x-app-request": "ip-check",
    },
    "body": {
        "env": {key: "[redacted fake value]" for key in fake_env},
    },
}
fake_request

## Evidence Boundary

The strongest inference is that the first-stage POST likely succeeded because
the loader received and executed a stage-two response. That is stronger than
mere capability but weaker than a complete server-side record of every field
received.

The public repository does not disclose the raw captured body. It does not
assert that every variable present in the victim shell was transmitted, and it
does not turn potential credential exposure into a claim of credential use.

In [ ]:
controls = [
    {"control": "clean-room review user", "breaks": "credential reachability"},
    {"control": "empty inherited environment", "breaks": "bulk secret capture"},
    {"control": "container or disposable VM", "breaks": "host-data reachability"},
    {"control": "egress deny by default", "breaks": "POST completion"},
    {"control": "install scripts disabled for inspection", "breaks": "automatic entrypoint"},
]
controls

**Observed:** the stage-two execution path was reached.

**Likely:** the environment POST completed sufficiently for the serverless
endpoint to return the captured stage.

**Not proven publicly:** the exact request body received by the remote service.

**Detection:** alert on Node processes posting a broad environment-shaped JSON
object, especially under npm ancestry and with unusual custom headers.

**Confidence upgrade:** endpoint logs, a packet capture with payload visibility,
or a cryptographically preserved request body.